# 🌍 Complete Geocoding Demo - Geocod.io

This notebook demonstrates how to geocode addresses using the **Geocod.io API** in Snowflake.

**What this notebook does:**
- Creates source and destination tables for address data
- Sets up external access to call the Geocod.io Geocoding API
- Provides a process to geocode addresses that haven't been processed yet

---

## 📋 Setup Instructions

Before running this notebook, you need to get a Geocod.io API key:

1. Go to [https://www.geocod.io/free-geocoding/](https://www.geocod.io/free-geocoding/) and create a free account
2. Once logged in, go to **API Keys** and create an API key
3. Select options for **GET Single geocoding** and **reverse geocoding endpoint**
4. Copy the API key and replace `'YOUR_API_KEY_HERE'` in the Secret creation cell below

---
## 🏗️ Step 1: Set Database and Schema Context

In [ ]:
USE ROLE ACCOUNTADMIN;

USE DATABASE DEMO_GEOCODE;
USE SCHEMA ADDRESS_PROCESSING;

---
## 🌐 Step 2: Create Network Rule for Geocod.io API

This network rule allows Snowflake to make outbound connections to the Geocod.io API.

In [ ]:
CREATE OR REPLACE NETWORK RULE geocodio_geocode_network_rule
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = ('api.geocod.io');

---
## 🔐 Step 3: Create Secret to Store API Key

⚠️ **IMPORTANT:** Replace `'YOUR_API_KEY_HERE'` with your actual Geocod.io API key!

In [ ]:
CREATE OR REPLACE SECRET geocodio_api_key_secret
TYPE = GENERIC_STRING 
--SECRET_STRING = 'YOUR_API_KEY_HERE';  -- Replace with your actual Geocod.io API key
SECRET_STRING = 'deeddf2497d217e184116627e164e8edded1899';  -- Replace with your actual Geocod.io API key


---
## 🔗 Step 4: Create External Access Integration

This integration combines the network rule and secret to enable secure API calls.

In [ ]:
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION geocodio_geocode_integration
  ALLOWED_NETWORK_RULES = (geocodio_geocode_network_rule)
  ALLOWED_AUTHENTICATION_SECRETS = (DEMO_GEOCODE.ADDRESS_PROCESSING.geocodio_api_key_secret)
  ENABLED = true;

---
## 🐍 Step 5: Create Python UDF to Call Geocod.io API

This User-Defined Function takes an address string and returns the geocoding result from the Geocod.io API.

In [ ]:
CREATE OR REPLACE FUNCTION geocodio_geocode_address(address_string STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.10
HANDLER = 'geocode_address'
EXTERNAL_ACCESS_INTEGRATIONS = (geocodio_geocode_integration)
PACKAGES = ('snowflake-snowpark-python','requests')
SECRETS = ('geocodio_api_key' = geocodio_api_key_secret)
AS
$$
import _snowflake
import requests
import json
import urllib.parse

session = requests.Session()

def geocode_address(address_string):
    try:
        api_key = _snowflake.get_generic_secret_string('geocodio_api_key')
        # URL encode the address
        encoded_address = urllib.parse.quote(address_string)
        
        # Geocod.io API URL
        url = f"https://api.geocod.io/v1.9/geocode?q={encoded_address}&api_key={api_key}"
        
        response = session.get(url)
        # Raise HTTP errors like 4xx or 5xx
        response.raise_for_status()
        return response.text
    except Exception as e:
        # Return structured error response
        return json.dumps({"error": str(e), "address": address_string})
$$;

### 🧪 Quick Test of the UDF

In [ ]:
SELECT geocodio_geocode_address('203 Autumn Rd Greer SC 29650');

---
## 📊 Step 6: Create Data Tables

### Source Addresses Table
Stores the original addresses to be geocoded.

In [ ]:
CREATE OR REPLACE TABLE Source_Addresses (
    Name VARCHAR(255),
    Department VARCHAR(100),
    Address_ID INTEGER AUTOINCREMENT,
    Address_Source_ID VARCHAR(100),
    Address VARCHAR(500),
    GeoCoded VARCHAR(3) DEFAULT 'No'
);

### Geocoded Addresses Table
Stores the geocoding results including latitude, longitude, and parsed address components.

In [ ]:
CREATE OR REPLACE TABLE Geocoded_Addresses (
    Geocoded_ID INTEGER AUTOINCREMENT,
    Name VARCHAR(255),
    Department VARCHAR(100),
    Address_ID INTEGER,
    Address_Source_ID VARCHAR(100),
    Address VARCHAR(500),
    Street VARCHAR(255),
    City VARCHAR(100),
    State VARCHAR(100),
    Zip VARCHAR(20),
    Lat FLOAT,
    Long FLOAT,
    Geocoded_Timestamp TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP() 
);

---
## 📝 Step 7: Insert Sample Addresses

In [ ]:
INSERT INTO Source_Addresses (Name, Address_Source_ID, Department, Address, GeoCoded) VALUES
    ('Miami Beach Property', 'A1', 'Sales', '4601 Collins Ave Miami Beach FL 33140', 'No'),
    ('Rochester Home', 'A2','Marketing', '114 Orland Rd Rochester NY 14622', 'No'),
    ('Greenville Residence','A3', 'Sales', '103 Autumn Rd Greenville SC 29650', 'No'),
    ('Clemson1', 'A4', 'Student', '217 W Main ST Central SC 29630', 'No'),
    ('Clemson2', 'A5', 'Student', '119 N Townville St Seneca SC 29678', 'No'),
    ('Clemson3', 'A6', 'Student', '356 Clemosn St Clemson SC 29631', 'No');

---
## ⚙️ Step 8: Create Stored Procedure for Geocoding

This procedure processes all addresses that haven't been geocoded yet using **Geocod.io's JSON structure**.

**Note:** Geocod.io returns results in a `results` array with `address_components` and `location` objects.

In [ ]:
CREATE OR REPLACE PROCEDURE GEOCODIO_Process_Ungeocoded_Addresses()
RETURNS STRING
LANGUAGE SQL
AS
$$
DECLARE
    processed_count INTEGER DEFAULT 0;
    error_count INTEGER DEFAULT 0;
    total_count INTEGER DEFAULT 0;
    result_message STRING;
BEGIN
    -- Get count of addresses to process
    SELECT COUNT(*) INTO total_count
    FROM Source_Addresses
    WHERE GeoCoded = 'No';
    
    -- Process each ungeocoded address
    INSERT INTO Geocoded_Addresses (Name, Department, Address_ID, Address_Source_ID, Address, Street, City, State, Zip, Lat, Long)
    WITH addresses_to_process AS (
        SELECT 
            Name,
            Department,
            Address_ID,
            Address_Source_ID, 
            Address
        FROM Source_Addresses
        WHERE GeoCoded = 'No'
    ),
    geocoded_data AS (
        SELECT
            a.Name,
            a.Department,
            a.Address_ID,
            a.Address_Source_ID, 
            a.Address,
            PARSE_JSON(geocodio_geocode_address(a.Address))::VARIANT AS api_response 
        FROM addresses_to_process a
    ),
    parsed_results AS (
        -- Geocod.io specific JSON parsing paths
        SELECT
            g.Name,
            g.Department,
            g.Address_ID,
            g.Address_Source_ID, 
            g.Address,
            -- Extract address components from the top result
            f.value:address_components.formatted_street::STRING AS street,
            f.value:address_components.city::STRING AS city,
            f.value:address_components.state::STRING AS state,
            f.value:address_components.zip::STRING AS zip,
            -- Extract coordinates from the location object
            f.value:location.lat::FLOAT AS lat,
            f.value:location.lng::FLOAT AS long
        FROM geocoded_data g,
        -- Geocod.io uses 'results' for its array
        LATERAL FLATTEN(input => g.api_response:results, outer => true) f
        WHERE f.INDEX = 0 -- Only take the top match
    )
    SELECT 
        Name,
        Department,
        Address_ID,
        Address_Source_ID, 
        Address,
        street,
        city,
        state,
        zip,
        lat,
        long
    FROM parsed_results
    WHERE lat IS NOT NULL; -- Only insert successful geocodes
    
    -- Get count of successfully processed addresses
    processed_count := SQLROWCOUNT;
    
    -- Update GeoCoded flag for successfully processed addresses
    UPDATE Source_Addresses
    SET GeoCoded = 'Yes'
    WHERE Address_ID IN (SELECT DISTINCT Address_ID FROM Geocoded_Addresses)
    AND GeoCoded = 'No';
    
    -- Calculate errors
    error_count := total_count - processed_count;
    
    -- Build result message
    result_message := 'Processing Complete. Total addresses to process: ' || total_count || 
                      ', Successfully geocoded: ' || processed_count || 
                      ', Errors: ' || error_count;
    
    RETURN result_message;
END;
$$;

---
## 👀 Step 9: View Addresses That Need Geocoding

In [ ]:
SELECT 
    Address_ID,
    Address_Source_ID, 
    Name,
    Department,
    Address,
    GeoCoded
FROM Source_Addresses
WHERE GeoCoded = 'No'
ORDER BY Address_ID;

---
## 🚀 Step 10: Execute the Geocoding Process

Run this cell to geocode all pending addresses.

In [ ]:
CALL GEOCODIO_Process_Ungeocoded_Addresses();

---
## 📋 Step 11: View Results

### View All Source Addresses and Their Status

In [ ]:
SELECT 
    Address_ID,
    Address_Source_ID, 
    Name,
    Department,
    Address,
    GeoCoded
FROM Source_Addresses
ORDER BY Address_ID;

### View All Geocoded Results

In [ ]:
SELECT 
    Geocoded_ID,
    Name,
    Department,
    Address_ID,
    Address_Source_ID, 
    Address,
    Street,
    City,
    State,
    Zip,
    Lat,
    Long,
    Geocoded_Timestamp
FROM Geocoded_Addresses
ORDER BY Address_SOURCE_ID, Address_ID;

### 🗺️ Test with Google Maps Links

In [ ]:
SELECT 
    Address,  
    'https://www.google.com/maps?q=' || Lat || ',' || Long AS Google_Maps_Link
FROM Geocoded_Addresses 
WHERE Geocoded_Timestamp IS NOT NULL 
LIMIT 5;

### View Joined Data (Source + Geocoded Results)

In [ ]:
SELECT 
    s.Address_ID,
    s.Address_Source_ID, 
    s.Name,
    s.Department,
    s.Address AS Original_Address,
    s.GeoCoded,
    g.Street,
    g.City,
    g.State,
    g.Zip,
    g.Lat,
    g.Long,
    g.Geocoded_Timestamp
FROM Source_Addresses s
LEFT JOIN Geocoded_Addresses g ON s.Address_ID = g.Address_ID
ORDER BY s.Address_ID;

---
## 🔍 Error Handling: Find Addresses That Failed to Geocode

In [ ]:
SELECT 
    s.Address_ID,
    s.Address_Source_ID, 
    s.Name,
    s.Department,
    s.Address,
    s.GeoCoded
FROM Source_Addresses s
LEFT JOIN Geocoded_Addresses g ON s.Address_ID = g.Address_ID
WHERE g.Address_ID IS NULL 
  AND s.GeoCoded = 'No'
ORDER BY s.Address_ID;

---
## 📈 Step 12: Create Monitoring Views

### Geocoding Status View

In [ ]:
CREATE OR REPLACE VIEW Geocoding_Status_View AS
SELECT 
    COUNT(*) as total_addresses,
    SUM(CASE WHEN GeoCoded = 'Yes' THEN 1 ELSE 0 END) as geocoded_count,
    SUM(CASE WHEN GeoCoded = 'No' THEN 1 ELSE 0 END) as pending_count,
    ROUND(100.0 * SUM(CASE WHEN GeoCoded = 'Yes' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) as success_rate_pct
FROM Source_Addresses;

SELECT * FROM Geocoding_Status_View;

---
## 📦 Step 13: Batch Processing Procedure

Create a procedure that processes a limited number of addresses at a time.

In [ ]:
CREATE OR REPLACE PROCEDURE Geocodio_Process_Ungeocoded_Addresses_Batch(batch_size INTEGER)
RETURNS STRING
LANGUAGE SQL
AS
$$
DECLARE
    processed_count INTEGER DEFAULT 0;
    total_to_process INTEGER DEFAULT 0;
    result_message STRING;
BEGIN
    -- Get count of addresses to process (limited by batch_size)
    SELECT COUNT(*) INTO total_to_process
    FROM (
        SELECT Address_ID
        FROM Source_Addresses
        WHERE GeoCoded = 'No'
        LIMIT :batch_size
    );
    
    -- Process batch of ungeocoded addresses
    INSERT INTO Geocoded_Addresses (Name, Department, Address_ID, Address_Source_ID, Address, Street, City, State, Zip, Lat, Long)
    WITH addresses_to_process AS (
        SELECT 
            Name,
            Department,
            Address_ID,
            Address_Source_ID, 
            Address
        FROM Source_Addresses
        WHERE GeoCoded = 'No'
        LIMIT :batch_size
    ),
    geocoded_data AS (
        SELECT
            a.Name,
            a.Department,
            a.Address_ID,
            a.Address_Source_ID, 
            a.Address,
            PARSE_JSON(geocodio_geocode_address(a.Address))::VARIANT AS api_response
        FROM addresses_to_process a
    ),
    parsed_results AS (
        SELECT
            g.Name,
            g.Department,
            g.Address_ID,
            g.Address_Source_ID, 
            g.Address,
            -- Geocod.io JSON paths from 'address_components'
            f.value:address_components.formatted_street::STRING AS street,
            f.value:address_components.city::STRING AS city,
            f.value:address_components.state::STRING AS state,
            f.value:address_components.zip::STRING AS zip,
            -- Geocod.io JSON paths from 'location'
            f.value:location.lat::FLOAT AS lat,
            f.value:location.lng::FLOAT AS long
        FROM geocoded_data g,
        LATERAL FLATTEN(input => g.api_response:results, outer => true) f
        WHERE f.INDEX = 0 -- Only take the top match
    )
    SELECT 
        Name,
        Department,
        Address_ID,
        Address_Source_ID, 
        Address,
        street,
        city,
        state,
        zip,
        lat,
        long
    FROM parsed_results
    WHERE lat IS NOT NULL;
    
    -- Get count of successfully processed addresses
    processed_count := SQLROWCOUNT;
    
    -- Update GeoCoded flag for successfully processed addresses
    UPDATE Source_Addresses
    SET GeoCoded = 'Yes'
    WHERE Address_ID IN (
        SELECT Address_ID 
        FROM Geocoded_Addresses 
        WHERE Address_ID NOT IN (
            SELECT Address_ID 
            FROM Source_Addresses 
            WHERE GeoCoded = 'Yes'
        )
    )
    AND GeoCoded = 'No';
    
    -- Build result message
    result_message := 'Batch Processing Complete. Total addresses to process: ' || total_to_process || 
                      ', Successfully geocoded: ' || processed_count || 
                      ', Errors: ' || (total_to_process - processed_count);
    
    RETURN result_message;
END;
$$;

### Example: Process in Batches of 100

In [ ]:
CALL Geocodio_Process_Ungeocoded_Addresses_Batch(100);

### Geocoding Analytics View

In [ ]:
CREATE OR REPLACE VIEW Geocoding_Analytics_View AS
SELECT 
    DATE_TRUNC('day', Geocoded_Timestamp) as geocoded_date,
    Department,
    COUNT(*) as addresses_geocoded, 
    COUNT(DISTINCT State) as states_covered,
    COUNT(DISTINCT City) as cities_covered,
    AVG(Lat) as avg_latitude,
    AVG(Long) as avg_longitude
FROM Geocoded_Addresses
GROUP BY DATE_TRUNC('day', Geocoded_Timestamp), Department
ORDER BY geocoded_date DESC, Department;

SELECT * FROM Geocoding_Analytics_View;

### Unprocessed Addresses View

In [ ]:
CREATE OR REPLACE VIEW Unprocessed_Addresses_View AS
SELECT 
    s.Address_ID,
    s.Address_Source_ID, 
    s.Name,
    s.Department,
    s.Address,
    s.GeoCoded
FROM Source_Addresses s
LEFT JOIN Geocoded_Addresses g ON s.Address_ID = g.Address_ID
WHERE g.Address_ID IS NULL 
  AND s.GeoCoded = 'No'
ORDER BY s.Address_ID;

SELECT * FROM Unprocessed_Addresses_View;

---
## 🏢 Step 14: Department-Specific Queries

### View Addresses by Department

In [ ]:
SELECT 
    Department,
    COUNT(*) as total_addresses,
    SUM(CASE WHEN GeoCoded = 'Yes' THEN 1 ELSE 0 END) as geocoded,
    SUM(CASE WHEN GeoCoded = 'No' THEN 1 ELSE 0 END) as pending
FROM Source_Addresses
GROUP BY Department
ORDER BY Department;

### View Geocoded Results for Sales Department

In [ ]:
SELECT 
    Name,
    Address,
    City,
    State,
    Zip,
    Lat,
    Long,
    Geocoded_Timestamp
FROM Geocoded_Addresses
WHERE Department = 'Sales'
ORDER BY Geocoded_Timestamp DESC;

### Get Marketing Addresses That Need Geocoding

In [ ]:
SELECT 
    Address_ID,
    Address_Source_ID, 
    Name,
    Address
FROM Source_Addresses
WHERE Department = 'Marketing'
  AND GeoCoded = 'No'
ORDER BY Address_ID;

### Summary Statistics by Department

In [ ]:
SELECT 
    Department,
    COUNT(*) as total_geocoded,
    COUNT(DISTINCT State) as states_covered,
    COUNT(DISTINCT City) as cities_covered,
    MIN(Geocoded_Timestamp) as first_geocoded,
    MAX(Geocoded_Timestamp) as last_geocoded
FROM Geocoded_Addresses
GROUP BY Department
ORDER BY total_geocoded DESC;

### Department Usage Over Time (Last 30 Days)

In [ ]:
SELECT 
    DATE(Geocoded_Timestamp) as geocoded_date,
    Department,
    COUNT(*) as addresses_geocoded
FROM Geocoded_Addresses
WHERE Geocoded_Timestamp >= DATEADD(day, -30, CURRENT_TIMESTAMP())
GROUP BY DATE(Geocoded_Timestamp), Department
ORDER BY geocoded_date DESC, Department;

### Create Department Summary View

In [ ]:
CREATE OR REPLACE VIEW Department_Summary_View AS
SELECT 
    s.Department,
    COUNT(DISTINCT s.Address_ID) as total_addresses,
    SUM(CASE WHEN s.GeoCoded = 'Yes' THEN 1 ELSE 0 END) as geocoded_count,
    SUM(CASE WHEN s.GeoCoded = 'No' THEN 1 ELSE 0 END) as pending_count,
    COUNT(DISTINCT g.State) as states_covered,
    COUNT(DISTINCT g.City) as cities_covered,
    MIN(g.Geocoded_Timestamp) as first_address_geocoded,
    MAX(g.Geocoded_Timestamp) as most_recent_geocoded
FROM Source_Addresses s
LEFT JOIN Geocoded_Addresses g ON s.Address_ID = g.Address_ID
GROUP BY s.Department
ORDER BY total_addresses DESC;

SELECT * FROM Department_Summary_View;

---
## ➕ Add More Addresses (Example)

Uncomment and run this cell to add more sample addresses, then run the geocoding process again.

In [ ]:
 INSERT INTO Source_Addresses (Name, Department, Address, GeoCoded) VALUES
     ('Test Address 1', 'Sales', '1 Market St San Francisco CA 94105', 'No'),
     ('Test Address 2', 'Marketing', '1060 W Addison St Chicago IL 60613', 'No');

-- Then run the process again
 CALL GEOCODIO_Process_Ungeocoded_Addresses();

---
## ⏰ Optional: Create Task for Scheduled Processing

Create an automated task to process addresses on a schedule.

In [ ]:
-- Create a task to run every hour (uncomment to create)
-- CREATE OR REPLACE TASK geocodio_geocode_addresses_task
--   WAREHOUSE = COMPUTE_WH  -- Change to your warehouse name
--   SCHEDULE = 'USING CRON 0 * * * * America/New_York'  -- Every hour
-- AS
--   CALL GEOCODIO_Process_Ungeocoded_Addresses();

-- Enable the task
-- ALTER TASK geocodio_geocode_addresses_task RESUME;

-- Check task status
-- SHOW TASKS LIKE 'geocodio_geocode_addresses_task';

-- View task history
-- SELECT *
-- FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY())
-- WHERE NAME = 'GEOCODIO_GEOCODE_ADDRESSES_TASK'
-- ORDER BY SCHEDULED_TIME DESC
-- LIMIT 10;

-- Suspend the task when not needed
-- ALTER TASK geocodio_geocode_addresses_task SUSPEND;

-- Drop the task
-- DROP TASK IF EXISTS geocodio_geocode_addresses_task;

---
## 🔄 Reset Demo

Use these commands to reset the demo data.

In [ ]:
-- Clear geocoded data and reset flags (uncomment to run)
-- DELETE FROM Geocoded_Addresses;
-- UPDATE Source_Addresses SET GeoCoded = 'No';

---
## 🗑️ Cleanup (Drop Everything)

⚠️ **WARNING:** This will delete all objects created by this notebook!

In [ ]:
-- Uncomment and run to drop all objects
-- DROP TASK IF EXISTS geocodio_geocode_addresses_task;
-- DROP VIEW IF EXISTS Department_Summary_View;
-- DROP VIEW IF EXISTS Unprocessed_Addresses_View;
-- DROP VIEW IF EXISTS Geocoding_Analytics_View;
-- DROP VIEW IF EXISTS Geocoding_Status_View;
-- DROP PROCEDURE IF EXISTS Geocodio_Process_Ungeocoded_Addresses_Batch(INTEGER);
-- DROP PROCEDURE IF EXISTS GEOCODIO_Process_Ungeocoded_Addresses();
-- DROP TABLE IF EXISTS Geocoded_Addresses;
-- DROP TABLE IF EXISTS Source_Addresses;
-- DROP FUNCTION IF EXISTS geocodio_geocode_address(STRING);
-- DROP EXTERNAL ACCESS INTEGRATION IF EXISTS geocodio_geocode_integration;
-- DROP SECRET IF EXISTS geocodio_api_key_secret;
-- DROP NETWORK RULE IF EXISTS geocodio_geocode_network_rule;